# 🔧 Notebook 02: Preprocesamiento de Datos

**Proyecto:** Predicción de duración de casos judiciales cerrados en Bolivia  
**Autores:** Vergara & Patiño  
**Materia:** Tecnologías Emergentes (TI26)  

Este notebook realiza la limpieza y transformación de los datos crudos usando PySpark, y exporta un CSV procesado listo para el modelado.

In [ ]:
# Instalar dependencias
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, datediff, when, lit

spark = SparkSession.builder.master("local[*]").appName("Preprocesamiento").getOrCreate()
print("Spark iniciado correctamente ✅")

## 1. Carga de Datos Crudos

In [ ]:
# Ruta del archivo (ruta local)
file_path = "../data/raw/CASOS_CERRADOS _publico__1.csv"
df = spark.read.csv(file_path, header=True, inferSchema=False)

print(f"Registros cargados: {df.count()}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

## 2. Limpieza de Fechas

In [ ]:
# Filtrar registros con formato de fecha válido
date_pattern = r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}$'
df_fechas = df.filter(
    col("denuncia_fecha_hora").rlike(date_pattern) &
    col("fecha_hora_cierre").rlike(date_pattern)
)

print(f"Registros con fechas válidas: {df_fechas.count()}")

## 3. Cálculo de Variable Objetivo (duracion_dias)

In [ ]:
df_clean = df_fechas.withColumn(
    "denuncia_ts", to_timestamp(col("denuncia_fecha_hora"), "dd-MM-yyyy HH:mm:ss")
).withColumn(
    "cierre_ts", to_timestamp(col("fecha_hora_cierre"), "dd-MM-yyyy HH:mm:ss")
).withColumn(
    "duracion_dias", datediff("cierre_ts", "denuncia_ts")
).filter(col("duracion_dias") >= 0)

print(f"Registros con duración >= 0 días: {df_clean.count()}")

## 4. Conversión a Columnas Numéricas

In [ ]:
# Cast numéricos
num_cols = ["victima_edad", "denunciado_edad", "hecho_gestion", "hecho_mes", "hecho_dia", "hecho_dia_semana"]
for c in num_cols:
    df_clean = df_clean.withColumn(c + "_num", col(c).cast("double"))

print("Columnas numéricas creadas ✅")
print("Columnas con sufijo _num:", [c + '_num' for c in num_cols])

## 5. Tratamiento de Nulos en Delito

In [ ]:
# Rellenar nulos en la columna delito
nulos_antes = df_clean.filter(col("delito").isNull()).count()
df_clean = df_clean.withColumn(
    "delito", when(col("delito").isNull(), lit("DESCONOCIDO")).otherwise(col("delito"))
)
nulos_despues = df_clean.filter(col("delito").isNull()).count()

print(f"Nulos en 'delito' antes: {nulos_antes}")
print(f"Nulos en 'delito' después: {nulos_despues}")

## 6. Selección de Columnas Finales

In [ ]:
# Seleccionar columnas finales para el modelado
final_cols = [c + "_num" for c in num_cols] + ["delito", "hecho_departamento", "duracion_dias"]
df_final = df_clean.select(*final_cols)

print(f"\n📋 Registros finales: {df_final.count()}")
print(f"📋 Columnas finales: {final_cols}")
df_final.printSchema()
df_final.show(5)

## 7. Exportación a CSV

In [ ]:
# CREACIÓN DE VARIABLE OBJETIVO PARA CLASIFICACIÓN
# 1 si el caso duró más de 365 días (largo/retardado), 0 si fue rápido
from pyspark.sql.functions import when, col
df_final = df_final.withColumn("caso_largo", when(col("duracion_dias") > 365, 1).otherwise(0))
print("Distribución de la nueva variable de clasificación:")
df_final.groupBy("caso_largo").count().show()
# Agregamos caso_largo a la lista de columnas finales antes de exportar
final_cols.append("caso_largo")


In [ ]:
# Exportar a CSV para los notebooks de modelado (03 y 04)
pandas_df = df_final.toPandas()
pandas_df.to_csv("../data/processed/casos_procesados.csv", index=False)

print(f"\n✅ Archivo 'casos_procesados.csv' generado exitosamente.")
print(f"   Filas: {len(pandas_df)}, Columnas: {len(pandas_df.columns)}")
print(f"\n📂 Descárgalo y colócalo en data/processed/ de tu repositorio.")
print(f"   Los notebooks 03 y 04 lo usarán como entrada.")

In [ ]:
# Verificar el archivo generado
import pandas as pd
check = pd.read_csv("../data/processed/casos_procesados.csv")
print(f"Verificación: {check.shape[0]} filas x {check.shape[1]} columnas")
check.head()

In [ ]:
# Cerrar sesión Spark
spark.stop()
print("\n🏁 Spark detenido. Preprocesamiento completado exitosamente.")